In [71]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='darkgrid')

from catboost import CatBoostClassifier
from sklearn.model_selection import GroupKFold, KFold
from sklearn.metrics import roc_auc_score

In [75]:
description = pd.read_excel('../data/description.xlsx')
X_train = pd.read_csv('../data/train_X.csv').fillna(0).drop(['contract_id' , 'contract_date', 'report_date'], axis=1)
X_test = pd.read_csv('../data/test1_X.csv').fillna(0).drop(['contract_id' , 'contract_date', 'report_date'], axis=1)
y_train = pd.read_csv('../data/train_y.csv')

In [79]:
contract_id = pd.read_csv('../data/train_X.csv')['contract_id'].values

In [77]:
y_train = y_train['default6']

In [78]:
def gini_score(y_true, y_pred):
    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1
    return gini


In [80]:
model = CatBoostClassifier(iterations=1000, 
                           learning_rate=0.1, 
                           depth=6, 
                           eval_metric='AUC',
                           random_seed=42,
                           verbose=0)

gkf = GroupKFold(n_splits=5)
gini_scores = []

for train_index, test_index in gkf.split(X_train, y_train, groups=contract_id):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model.fit(X_train_fold, y_train_fold, eval_set=(X_test_fold, y_test_fold), early_stopping_rounds=100, use_best_model=True)
    
    y_pred = model.predict_proba(X_test_fold)[:, 1]
    
    gini = gini_score(y_test_fold, y_pred)
    gini_scores.append(gini)
    print(f'Fold Gini Score: {gini}')

mean_gini_score = np.mean(gini_scores)
print(f'Mean Gini Score: {mean_gini_score}')


Fold Gini Score: 0.502895449586575
Fold Gini Score: 0.5384471681788987
Fold Gini Score: 0.5594243711614315
Fold Gini Score: 0.5574397661709221
Fold Gini Score: 0.5922565124933548
Mean Gini Score: 0.5500926535182364


In [74]:
model = CatBoostClassifier(iterations=1000, 
                           learning_rate=0.1, 
                           depth=6, 
                           eval_metric='AUC',
                           random_seed=42,
                           verbose=0)

gkf = KFold(n_splits=5, shuffle=False)
gini_scores = []

for train_index, test_index in gkf.split(X_train, y_train):
    X_train_fold, X_test_fold = X_train.iloc[train_index], X_train.iloc[test_index]
    y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
    
    model.fit(X_train_fold, y_train_fold, eval_set=(X_test_fold, y_test_fold), early_stopping_rounds=100, use_best_model=True)
    
    y_pred = model.predict_proba(X_test_fold)[:, 1]
    
    gini = gini_score(y_test_fold, y_pred)
    gini_scores.append(gini)
    print(f'Fold Gini Score: {gini}')

mean_gini_score = np.mean(gini_scores)
print(f'Mean Gini Score: {mean_gini_score}')


Fold Gini Score: 0.8525112193965965
Fold Gini Score: 0.9861186173478025
Fold Gini Score: 0.9780635965602738
Fold Gini Score: 0.9869752311816349
Fold Gini Score: 0.8551807423563149
Mean Gini Score: 0.9317698813685246


In [ ]:
pd.Series(X_test['project_id'].unique()).isin(X_train['project_id'].unique()).sum()